# North Carolina Energy Analysis
## Project Overview

This project analyzes electricity demand in North Carolina and explores how weather patterns influence energy usage. Using data from the U.S. Energy Information Administration (EIA) and Open-Meteo, the project collects hourly and daily electricity demand along with historical weather data. The data is cleaned, merged, and analyzed to identify relationships between temperature, weather conditions, and electricity demand. The long-term goal is to build interactive visualizations and dashboards that provide insights into energy consumption trends across North Carolina.

In [103]:
import pandas as pd
import requests as rq
import json
import datetime as dt
import prettytable
import sqlite3 as sql3
import plotly.express as px
from dotenv import load_dotenv
import os
import openmeteo_requests
import requests_cache
from retry_requests import retry
import numpy as np

load_dotenv()

EIA_API_KEY = os.getenv("EIA_API_KEY")
print(1)

1


## Data Collection

In [106]:
prettytable.DEFAULT = 'DEFAULT'
conn = sql3.connect("North_Carolina_Energy_DB.db")
cursor = conn.cursor()
%load_ext sql
%sql sqlite:///North_Carolina_Energy_DB.db

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [108]:
today = dt.datetime.now()
from_date = today - dt.timedelta(days=30)

In [110]:
eia_start = from_date.strftime("%Y-%m-%d")
eia_end = dt.datetime.now().strftime("%Y-%m-%d")
eia_api_length = 720
eia_hourly_start = ""
eia_hourly_end = ""
eia_hourly_length = 720

### Energy Demand

#### Daily Demand

In [114]:
eia_daily_url = f"https://api.eia.gov/v2/electricity/rto/daily-region-data/data/?frequency=daily&data[0]=value&facets[respondent][]=CAR&start={eia_start}&end={eia_end}&sort[0][column]=period&sort[0][direction]=desc&offset=0&length={eia_api_length}&api_key={EIA_API_KEY}"
eia_daily_response = rq.get(eia_daily_url)
daily_data = eia_daily_response.json()
eia_daily_data = daily_data['response']['data']

#### Hourly Demand

In [116]:
eia_hourly_url = f"https://api.eia.gov/v2/electricity/rto/region-data/data/?frequency=hourly&data[0]=value&facets[type][]=D&facets[respondent][]=CAR&start={eia_start}&end={eia_end}&sort[0][column]=period&sort[0][direction]=desc&offset=0&length={eia_hourly_length}&api_key={EIA_API_KEY}"
eia_hourly_response = rq.get(eia_hourly_url)
hourly_data = eia_hourly_response.json()
eia_hourly_data = hourly_data['response']['data']

### Weather

#### Daily Weather

In [119]:

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
daily_weather_url = "https://archive-api.open-meteo.com/v1/archive"
daily_params = {
	"latitude": [35.5951, 36.0999, 35.2271, 35.7796, 34.2104, 34.8526, 34.0007, 34.1954, 32.7765, 33.6891],
	"longitude": [-82.5515, -80.2442, -80.8431, -78.6382, -77.8868, -82.3940, -81.0348, -79.7626, -79.9311, -78.8867],
	"start_date": eia_start,
	"end_date": eia_end,
    "daily": ["temperature_2m_max", "temperature_2m_min", "wind_speed_10m_max", "wind_speed_10m_min", "precipitation_sum"],
    "temperature_unit": "fahrenheit",
    "wind_speed_unit": "mph",
    "precipitation_unit": "inch"
}
daily_weather_responses = openmeteo.weather_api(daily_weather_url, params = daily_params)

#### Hourly Weather

In [121]:
# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
hourly_weather_url = "https://archive-api.open-meteo.com/v1/archive"
hourly_params = {
	"latitude": 35.7796,
	"longitude": -78.6382,
	"start_date": eia_start,
	"end_date": eia_end,
    "hourly": ["temperature_2m", "relative_humidity_2m", "wind_speed_10m", "cloud_cover", "precipitation"],
    "temperature_unit": "fahrenheit",
    "wind_speed_unit": "mph",
    "precipitation_unit": "inch"
}
hourly_weather_responses = openmeteo.weather_api(hourly_weather_url, params = hourly_params)

### Energy Generation

#### Daily Energy Generation

In [125]:
eia_daily_generation_url = f"https://api.eia.gov/v2/electricity/rto/daily-fuel-type-data/data/?frequency=daily&data[0]=value&facets[fueltype][]=COL&facets[fueltype][]=NG&facets[fueltype][]=NUC&facets[fueltype][]=OIL&facets[fueltype][]=OTH&facets[fueltype][]=SUN&facets[fueltype][]=WAT&facets[fueltype][]=WND&facets[respondent][]=CAR&facets[timezone][]=Eastern&start={eia_start}&end={eia_end}&sort[0][column]=period&sort[0][direction]=desc&offset=0&length=5000&api_key={EIA_API_KEY}"
eia_daily_generation_response = rq.get(eia_daily_generation_url)
daily_generation_data = eia_daily_generation_response.json()
eia_daily_generation_data = daily_generation_data['response']['data']

#### Hourly Energy Generation

In [127]:
eia_hourly_generation_url = f"https://api.eia.gov/v2/electricity/rto/fuel-type-data/data/?frequency=hourly&data[0]=value&facets[fueltype][]=COL&facets[fueltype][]=NG&facets[fueltype][]=NUC&facets[fueltype][]=OIL&facets[fueltype][]=OTH&facets[fueltype][]=SUN&facets[fueltype][]=WAT&facets[fueltype][]=WND&facets[respondent][]=CAR&start={eia_start}&end={eia_end}&sort[0][column]=period&sort[0][direction]=desc&offset=0&length=5000&api_key={EIA_API_KEY}"
eia_hourly_generation_response = rq.get(eia_hourly_generation_url)
hourly_generation_data = eia_hourly_generation_response.json()
eia_hourly_generation_data = hourly_generation_data['response']['data']

## Data Cleaning

### Energy Demand

#### Daily Energy Demand

In [132]:
d_rows = []
for item in eia_daily_data:
    if item['type'] == "D" and item['timezone'] == "Eastern":
        d_rows.append({
            "Date": pd.to_datetime(item["period"]).date(),
            "Demand (MWh)": item["value"]
    })
eia_daily_df = pd.DataFrame(d_rows)
eia_daily_df.dropna(inplace=True)

#### Hourly Demand

In [135]:
h_rows = []
for item in eia_hourly_data:
    h_rows.append({
        "Datetime": pd.to_datetime(item["period"]),
        "Date": pd.to_datetime(item["period"]).date(),
        "Hour": pd.to_datetime(item["period"]).hour,
        "Demand (MWh)": item["value"]
    })
eia_hourly_df = pd.DataFrame(h_rows)
eia_hourly_df.dropna(inplace=True)

### Weather

#### Daily Weather

In [139]:
# Process daily data. The order of variables needs to be the same as requested.
high_temp = []
low_temp = []
mx_w_s = []
mn_w_s = []
prcp_sum = []

for response in daily_weather_responses:
    daily = response.Daily()
    daily_temperature_2m_max = daily.Variables(0).ValuesAsNumpy()
    daily_temperature_2m_min = daily.Variables(1).ValuesAsNumpy()
    daily_wind_speed_10m_max = daily.Variables(2).ValuesAsNumpy()
    daily_wind_speed_10m_min = daily.Variables(3).ValuesAsNumpy()
    daily_precipitation_sum = daily.Variables(4).ValuesAsNumpy()
    high_temp.append(daily_temperature_2m_max)
    low_temp.append(daily_temperature_2m_min)
    mx_w_s.append(daily_wind_speed_10m_max)
    mn_w_s.append(daily_wind_speed_10m_min)
    prcp_sum.append(daily_precipitation_sum)

daily_data = {
	"Date": pd.date_range(
		start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = daily.Interval()),
		inclusive = "left"
	).date,
    "High temp": np.mean(high_temp, axis=0).astype(int),
    "Low temp": np.mean(low_temp, axis=0).astype(int),
    "Max wind speed (mph)": np.round(np.mean(mx_w_s, axis=0).astype(float),2),
    "Min wind speed (mph)": np.round(np.mean(mn_w_s, axis=0).astype(float),2),  
    "Precipitation sum (inches)": np.round(np.mean(prcp_sum).astype(float),2)
}

daily_weather_df = pd.DataFrame(data = daily_data)
daily_weather_df

,Date,High temp,Low temp,Max wind speed (mph),Min wind speed (mph),Precipitation sum (inches)
0,2026-06-23,88,74,15.22,5.44,0.15
1,2026-06-24,83,65,11.33,2.64,0.15
2,2026-06-25,87,66,7.33,0.95,0.15
3,2026-06-26,88,70,9.87,1.64,0.15
4,2026-06-27,91,72,11.02,4.05,0.15
5,2026-06-28,90,73,10.71,3.83,0.15
6,2026-06-29,90,72,8.77,1.74,0.15
7,2026-06-30,90,72,8.35,2.19,0.15
8,2026-07-01,91,72,7.64,2.26,0.15
9,2026-07-02,94,73,7.68,1.13,0.15


#### Hourly Weather

In [141]:
# Process hourly data. The order of variables needs to be the same as requested.
hourly_temp = []
hourly_hum = []
hourly_w_s = []
hourly_cl_c = []
hourly_precip = []

for response in hourly_weather_responses:
    hourly = response.Hourly()
    hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
    hourly_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
    hourly_wind_speed_10m = hourly.Variables(2).ValuesAsNumpy()
    hourly_cloud_cover = hourly.Variables(3).ValuesAsNumpy()
    hourly_precipitation = hourly.Variables(4).ValuesAsNumpy()
    hourly_temp.append(hourly_temperature_2m)
    hourly_hum.append(hourly_humidity_2m)
    hourly_w_s.append(hourly_wind_speed_10m)
    hourly_cl_c.append(hourly_cloud_cover)
    hourly_precip.append(hourly_precipitation)


hourly_data = {
	"Date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	).date,
    "Hour": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	).hour,
    "Temperature": np.mean(hourly_temp, axis=0).astype(int),
    "Humidity %": np.round(np.mean(hourly_hum, axis=0).astype(float),2),
    "Wind speed (mph)": np.round(np.mean(hourly_w_s, axis=0).astype(float),2),
    "Cloud cover %": np.round(np.mean(hourly_cl_c, axis=0).astype(float), 2),
    "Precipitation (inches)": np.round(np.mean(hourly_precip, axis=0).astype(float),2)
}

hourly_weather_df = pd.DataFrame(data = hourly_data)

### Energy Generation

#### Daily Energy Generation

In [145]:
d_e_g_rows = []
for item in eia_daily_generation_data:
    d_e_g_rows.append({
        "Date": pd.to_datetime(item["period"]).date(),
        "Energy Type": item["type-name"],
        "Energy Generated (MWh)": item["value"]
    })
eia_daily_generation_df = pd.DataFrame(d_e_g_rows)
eia_daily_generation_df.dropna(inplace=True)

#### Hourly Energy Generation

In [147]:
h_e_g_rows = []
for item in eia_hourly_generation_data:
    d_e_g_rows.append({
        "Date": pd.to_datetime(item["period"]).date(),
        "Hour": pd.to_datetime(item["period"]).hour,
        "Energy Type": item["type-name"],
        "Energy Generated (MWh)": item["value"]
    })
eia_hourly_generation_df = pd.DataFrame(d_e_g_rows)
eia_hourly_generation_df.dropna(inplace=True)

## Database Creation

### Energy Demand

#### Daily Demand

In [156]:
eia_daily_df.to_sql("NC_DAILY_ELECTRICITY_DEMAND", conn, if_exists='replace', index=False, method='multi');
%sql sqlite:///North_Carolina_Energy_DB.db

#### Hourly Demand

In [159]:
eia_hourly_df.to_sql("NC_HOURLY_ELECTRICITY_DEMAND", conn, if_exists='replace', index=False, method='multi');
%sql sqlite:///North_Carolina_Energy_DB.db

### Weather
#### Daily Weather

In [162]:
daily_weather_df.to_sql("NC_DAILY_WEATHER_DATA", conn, if_exists='replace', index=False, method='multi');

#### Hourly Weather

In [164]:
hourly_weather_df.to_sql("NC_HOURLY_WEATHER_DATA", conn, if_exists='replace', index=False, method='multi');

### Energy Generation

#### Daily Energy Generation

In [167]:
eia_daily_generation_df.to_sql("NC_DAILY_ENERGY_GENERATION_DATA", conn, if_exists='replace', index=False, method='multi');

#### Hourly Energy Generation

In [169]:
eia_hourly_generation_df.to_sql("NC_HOURLY_ENERGY_GENERATION_DATA", conn, if_exists='replace', index=False, method='multi');

## SQL Analysis

### Average Hourly Demand

In [173]:
# Find the mean value for Hourly Demand
%sql select hour, round(avg([Demand (MWh)]), 2) as [Average MWh Demand Per Hour] from NC_HOURLY_ELECTRICITY_DEMAND group by hour;

 * sqlite:///North_Carolina_Energy_DB.db
Done.


Hour,Average MWh Demand Per Hour
0,38667.3
1,36981.27
2,35400.1
3,32978.97
4,30413.13
5,28185.4
6,26516.93
7,25275.73
8,24436.67
9,24114.57


### Average Daily Weather Attributes of the Last 30 Days  

In [181]:
%%sql 
select 
    round(avg([high temp]),2) as avg_high_temp, 
    round(avg([low temp]),2) as avg_low_temp, 
    round(avg([max wind speed (mph)]),2) as estimated_avg_max_wind_speed, 
    round(avg([min wind speed (mph)]),2) as estimated_avg_min_wind_speed, 
    round(avg([precipitation sum (inches)]),2) as avg_precipitation_in_inches  
from NC_DAILY_WEATHER_DATA;

 * sqlite:///North_Carolina_Energy_DB.db
Done.


avg_high_temp,avg_low_temp,estimated_avg_max_wind_speed,estimated_avg_min_wind_speed,avg_precipitation_in_inches
90.65,72.97,10.08,2.74,0.15


### Average Hourly Weather Attributes of the Last 30 Days  

In [225]:
%%sql 
select
    hour,
    round(avg([temperature]),2) as avg_temp, 
    round(avg("humidity %"),2) as "avg humidity %", 
    round(avg([wind speed (mph)]),2) as true_wind_speed, 
    round(avg("cloud cover %"),2) as "avg clound_cover %", 
    round(avg([precipitation (inches)]),2) as avg_precipitation_in_inches  
from NC_HOURLY_WEATHER_DATA group by hour;

 * sqlite:///North_Carolina_Energy_DB.db
Done.


Hour,avg_temp,avg humidity %,true_wind_speed,avg clound_cover %,avg_precipitation_in_inches
0,82.0,69.8,5.01,51.94,0.01
1,79.68,74.85,5.63,55.45,0.01
2,78.45,77.27,5.3,55.52,0.01
3,77.39,78.77,5.47,48.0,0.0
4,76.26,80.67,5.46,44.87,0.01
5,75.19,83.07,5.31,41.58,0.01
6,73.32,88.11,5.14,47.58,0.0
7,73.23,88.54,5.06,48.45,0.0
8,72.94,89.27,5.3,44.65,0.0
9,72.65,90.04,5.18,41.77,0.0


### Total Daily Energy Generated by Resource in last 30 Days

In [227]:
%%sql
select
    [Energy Type],
    sum([Energy Generated (MWh)]) as "Total Energy Generated (MWh) by Resource"
from NC_DAILY_ENERGY_GENERATION_DATA group by [Energy Type] order by "Total Energy Generated (MWh) by Resource" desc;


 * sqlite:///North_Carolina_Energy_DB.db
Done.


Energy Type,Total Energy Generated (MWh) by Resource
Nuclear,8480638
Natural Gas,6311321
Coal,6226469
Solar,1164287
Other,786557
Hydro,154817
Petroleum,263


### Total Hourly Energy Generated by Resource in last 30 Days

In [190]:
%%sql 
select
    distinct[Energy Type],
    round(avg([Energy Generated (MWh)]),2) as "Hourly Average Energy Generated (MWh) by Resource" 
from NC_HOURLY_ENERGY_GENERATION_DATA group by [Energy Type] order by "Hourly Average Energy Generated (MWh) by Resource" desc;

 * sqlite:///North_Carolina_Energy_DB.db
Done.


Energy Type,Hourly Average Energy Generated (MWh) by Resource
Nuclear,11778.03
Natural Gas,8786.34
Coal,8661.83
Solar,1629.38
Other,1091.31
Hydro,216.47
Petroleum,0.37


### Daily Energy Supply and Demand Sorted by the Difference

#### Daily Supply and Demand View Creation

In [409]:
%%sql
drop view if exists [Daily Supply and Demand];
create view [Daily Supply and Demand] as
select 
    d.Date,
    case strftime('%w', d.Date)
        when '0' then 'Sunday'
        when '1' then 'Monday'
        when '2' then 'Tuesday'
        when '3' then 'Wednesday'
        when '4' then 'Thursday'
        when '5' then 'Friday'
        when '6' then 'Saturday'
    end as [Day of Week],
    d.[Demand (MWh)], 
    sum(g.[Energy Generated (MWh)]) as [Total Daily Energy Generation],
    sum(g.[Energy Generated (MWh)]) - d.[Demand (MWh)] as Difference,
    round(cast(sum(g.[Energy Generated (MWh)]) - d.[Demand (MWh)]  as float) / d.[Demand (MWh)] * 100, 2) || '%' as [Percent Difference]
from NC_DAILY_ELECTRICITY_DEMAND as d
join NC_DAILY_ENERGY_GENERATION_DATA as g on d.Date = g.Date group by d.Date;
    

 * sqlite:///North_Carolina_Energy_DB.db
Done.
Done.


[]

In [411]:
%sql select * from [Daily Supply and Demand] order by Difference desc limit 5;

 * sqlite:///North_Carolina_Energy_DB.db
Done.


Date,Day of Week,Demand (MWh),Total Daily Energy Generation,Difference,Percent Difference
2026-07-14,Tuesday,714522,740147,25625,3.59%
2026-06-24,Wednesday,668643,690188,21545,3.22%
2026-07-13,Monday,691126,703384,12258,1.77%
2026-07-11,Saturday,736579,743267,6688,0.91%
2026-07-15,Wednesday,775701,779873,4172,0.54%


In [413]:
%sql PRAGMA table_info(NC_DAILY_WEATHER_DATA);

 * sqlite:///North_Carolina_Energy_DB.db
Done.


cid,name,type,notnull,dflt_value,pk
0,Date,DATE,0,None,0
1,High temp,INTEGER,0,None,0
2,Low temp,INTEGER,0,None,0
3,Max wind speed (mph),REAL,0,None,0
4,Min wind speed (mph),REAL,0,None,0
5,Precipitation sum (inches),REAL,0,None,0


### Supply and Demand Joined with Weather Table

In [419]:
%%sql 
select 
    s.*,
    w.[High temp],
    w.[Low temp],
    w.[Max wind speed (mph)],
    w.[Min wind speed (mph)],
    w.[Precipitation sum (inches)]	
from "Daily Supply and Demand" as s 
join [NC_DAILY_WEATHER_DATA] as w 
    on s.Date=w.Date where w.[High temp] > 85 group by s.Date order by w.Date desc;

 * sqlite:///North_Carolina_Energy_DB.db
Done.


Date,Day of Week,Demand (MWh),Total Daily Energy Generation,Difference,Percent Difference,High temp,Low temp,Max wind speed (mph),Min wind speed (mph),Precipitation sum (inches)
2026-07-22,Wednesday,843189,823239,-19950,-2.37%,92,77,14.45,6.52,0.15
2026-07-21,Tuesday,833259,810201,-23058,-2.77%,92,75,13.68,4.01,0.15
2026-07-20,Monday,829771,825659,-4112,-0.5%,91,74,9.77,2.85,0.15
2026-07-19,Sunday,812682,809565,-3117,-0.38%,93,76,10.99,4.75,0.15
2026-07-18,Saturday,830998,821281,-9717,-1.17%,92,76,12.07,1.9,0.15
2026-07-17,Friday,868907,826764,-42143,-4.85%,96,75,8.64,1.27,0.15
2026-07-16,Thursday,836770,806936,-29834,-3.57%,94,73,7.79,1.27,0.15
2026-07-15,Wednesday,775701,779873,4172,0.54%,90,70,6.39,1.12,0.15
2026-07-12,Sunday,720771,714180,-6591,-0.91%,89,72,8.55,1.61,0.15
2026-07-11,Saturday,736579,743267,6688,0.91%,90,73,11.12,4.36,0.15


### Weekend Supply and Demand with Weather Table

In [417]:
%%sql 
select 
    s.*,
    w.[High temp],
    w.[Low temp],
    w.[Max wind speed (mph)],
    w.[Min wind speed (mph)],
    w.[Precipitation sum (inches)]	
from "Daily Supply and Demand" as s
join [NC_DAILY_WEATHER_DATA] as w on s.Date=w.Date 
where s.[Day of Week] in ('Sunday', 'Saturday')
group by s.Date order by w.Date desc;


 * sqlite:///North_Carolina_Energy_DB.db
Done.


Date,Day of Week,Demand (MWh),Total Daily Energy Generation,Difference,Percent Difference,High temp,Low temp,Max wind speed (mph),Min wind speed (mph),Precipitation sum (inches)
2026-07-19,Sunday,812682,809565,-3117,-0.38%,93,76,10.99,4.75,0.15
2026-07-18,Saturday,830998,821281,-9717,-1.17%,92,76,12.07,1.9,0.15
2026-07-12,Sunday,720771,714180,-6591,-0.91%,89,72,8.55,1.61,0.15
2026-07-11,Saturday,736579,743267,6688,0.91%,90,73,11.12,4.36,0.15
2026-07-05,Sunday,773992,751078,-22914,-2.96%,94,75,10.47,1.83,0.15
2026-07-04,Saturday,817849,797205,-20644,-2.52%,96,75,8.81,1.73,0.15
2026-06-28,Sunday,716304,710138,-6166,-0.86%,90,73,10.71,3.83,0.15
2026-06-27,Saturday,721762,718108,-3654,-0.51%,91,72,11.02,4.05,0.15


### Weekday Supply and Demand with Weather Table

In [422]:
%%sql 
select 
    s.*,
    w.[High temp],
    w.[Low temp],
    w.[Max wind speed (mph)],
    w.[Min wind speed (mph)],
    w.[Precipitation sum (inches)]	
from "Daily Supply and Demand" as s
join [NC_DAILY_WEATHER_DATA] as w on s.Date=w.Date 
where s.[Day of Week] in ('Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday')
group by s.Date order by w.Date desc;

 * sqlite:///North_Carolina_Energy_DB.db
Done.


Date,Day of Week,Demand (MWh),Total Daily Energy Generation,Difference,Percent Difference,High temp,Low temp,Max wind speed (mph),Min wind speed (mph),Precipitation sum (inches)
2026-07-22,Wednesday,843189,823239,-19950,-2.37%,92,77,14.45,6.52,0.15
2026-07-21,Tuesday,833259,810201,-23058,-2.77%,92,75,13.68,4.01,0.15
2026-07-20,Monday,829771,825659,-4112,-0.5%,91,74,9.77,2.85,0.15
2026-07-17,Friday,868907,826764,-42143,-4.85%,96,75,8.64,1.27,0.15
2026-07-16,Thursday,836770,806936,-29834,-3.57%,94,73,7.79,1.27,0.15
2026-07-15,Wednesday,775701,779873,4172,0.54%,90,70,6.39,1.12,0.15
2026-07-14,Tuesday,714522,740147,25625,3.59%,83,70,9.71,3.38,0.15
2026-07-13,Monday,691126,703384,12258,1.77%,82,71,8.88,1.67,0.15
2026-07-10,Friday,799446,777169,-22277,-2.79%,94,74,12.64,5.3,0.15
2026-07-09,Thursday,845335,821779,-23556,-2.79%,94,76,11.37,4.54,0.15


In [ ]:
Overall demand vs. generation over time
    Line chart
    Shows how closely supply matches demand.
Largest surplus and deficit days
    Bar chart
    Identifies the biggest outliers.
Temperature vs. demand
    Scatter plot
    Tests whether hotter (or colder) days correspond to higher electricity demand.
Weather vs. generation deficit
    Scatter plot or grouped bar chart
    Looks at wind, precipitation, or temperature versus the percentage difference.
Weekday vs. weekend demand
    Bar chart
    Compares average demand by day type.
Seasonal demand
    Bar or line chart
    Spring, Summer, Fall, Winter.
Generation mix
    Stacked bar chart or pie chart for selected days
    Shows how much comes from nuclear, natural gas, coal, solar, and hydro.
Energy source response to weather
    Example: Natural gas generation vs. temperature.
    Example: Solar generation vs. precipitation or cloudiness (if you have those variables).